In [ ]:
# --- STEP 1: Install Dependencies ---
!pip install -q google-generativeai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 10.3 MB/s eta 0:00:00


In [ ]:
import google.generativeai as genai
import json
import time
from pypdf import PdfReader
from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:

# SETUP: Replace 'YOUR_API_KEY' with your actual Google AI Studio key
# Or set it in Colab secrets as 'GOOGLE_API_KEY'
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except:
    GOOGLE_API_KEY = "URKEY"

genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash') # Fast & Efficient

In [ ]:

# --- STEP 3: Define Helper Functions ---

def get_page_text(pdf_reader, page_num):
    """Extracts text from a specific page."""
    try:
        page = pdf_reader.pages[page_num]
        return page.extract_text()[:2000] # Limit text to save tokens
    except:
        return ""

def classify_document_type(text):
    """Determines the type of document based on its content."""
    prompt = f"""
    Analyze the following document text and classify it into ONE of these categories:
    [Resume, Bank Statement, Invoice, Contract, Lender Fee Sheet, Tax Form, ID Card, Utility Bill].

    Return ONLY the category name.

    Text:
    {text[:1000]}
    """
    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except:
        return "Unknown"

def is_same_document(prev_text, curr_text):
    """Decides if the current page belongs to the same document as the previous page."""
    prompt = f"""
    I will provide text from two consecutive pages of a PDF file.
    Your job is to determine if "Page 2" is a continuation of "Page 1" (part of the same document) or if "Page 2" is the start of a completely new, unrelated document.

    Answer ONLY with "YES" (same document) or "NO" (new document).

    --- Page 1 Text ---
    {prev_text[-500:]}

    --- Page 2 Text ---
    {curr_text[:500]}
    """
    try:
        response = model.generate_content(prompt)
        return "YES" in response.text.upper()
    except:
        return False

In [ ]:

# --- STEP 4: Main Execution Loop ---

pdf_path = "/content/TestBlobFile.pdf" # Make sure this matches your uploaded file name
reader = PdfReader(pdf_path)
total_pages = len(reader.pages)

results = []
current_doc_id = 0
current_doc_type = "Unknown"
previous_page_text = ""

print(f"Processing {total_pages} pages...")

for i in range(total_pages):
    print(f"Analyzing Page {i}...")
    current_page_text = get_page_text(reader, i)

    # Logic for Page 0 (Always the start)
    if i == 0:
        current_doc_type = classify_document_type(current_page_text)
        is_continuation = False
    else:
        # Compare with previous page
        is_continuation = is_same_document(previous_page_text, current_page_text)

        if not is_continuation:
            # It's a new document! Increment ID and re-classify
            current_doc_id += 1
            current_doc_type = classify_document_type(current_page_text)
            print(f"   -> Detected NEW Document: {current_doc_type}")
        else:
            print(f"   -> Continuation of Document {current_doc_id}")

    # Store Metadata
    page_metadata = {
        "page": i,
        "doc_id": current_doc_id,
        "doc_type": current_doc_type
    }
    results.append(page_metadata)

    # Update previous text for next loop
    previous_page_text = current_page_text
    time.sleep(1) # Avoid hitting API rate limits

Processing 7 pages...
Analyzing Page 0...
Analyzing Page 1...
   -> Detected NEW Document: Tax Form
Analyzing Page 2...
   -> Detected NEW Document: Contract
Analyzing Page 3...
   -> Continuation of Document 2
Analyzing Page 4...


   -> Detected NEW Document: Unknown
Analyzing Page 5...


   -> Detected NEW Document: Unknown
Analyzing Page 6...


   -> Detected NEW Document: Unknown


In [ ]:
# --- STEP 5: Export to JSON ---
output_file = "page_metadata.json"
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSuccess! Results saved to {output_file}")
print(json.dumps(results[:3], indent=2)) # Print preview


Success! Results saved to page_metadata.json
[
  {
    "page": 0,
    "doc_id": 0,
    "doc_type": "Lender Fee Sheet"
  },
  {
    "page": 1,
    "doc_id": 1,
    "doc_type": "Tax Form"
  },
  {
    "page": 2,
    "doc_id": 2,
    "doc_type": "Contract"
  }
]
